In [0]:
%sql
-- Create catalog for the project
CREATE CATALOG IF NOT EXISTS olist_project;

-- Create schemas for medallion architecture
CREATE SCHEMA IF NOT EXISTS olist_project.raw;
CREATE SCHEMA IF NOT EXISTS olist_project.bronze;
CREATE SCHEMA IF NOT EXISTS olist_project.silver;
CREATE SCHEMA IF NOT EXISTS olist_project.gold;

-- Create volume for raw CSV files
CREATE VOLUME IF NOT EXISTS olist_project.raw.olist_volume;

In [0]:
volume_path = "/Volumes/olist_project/raw/olist_volume/"

display(dbutils.fs.ls(volume_path))

In [0]:
base_path = "/Volumes/olist_project/raw/olist_volume/"

orders_bronze = spark.read.option("header", True).option("inferSchema", True).csv(
    base_path + "olist_orders_dataset.csv"
)

customers_bronze = spark.read.option("header", True).option("inferSchema", True).csv(
    base_path + "olist_customers_dataset.csv"
)

items_bronze = spark.read.option("header", True).option("inferSchema", True).csv(
    base_path + "olist_order_items_dataset.csv"
)

In [0]:
orders_bronze.write.format("delta").mode("overwrite").saveAsTable(
    "olist_project.bronze.orders"
)

customers_bronze.write.format("delta").mode("overwrite").saveAsTable(
    "olist_project.bronze.customers"
)

items_bronze.write.format("delta").mode("overwrite").saveAsTable(
    "olist_project.bronze.order_items"
)

### Load Bronze tables into Spark **DataFrames**

In [0]:
orders = spark.table("olist_project.bronze.orders")
customers = spark.table("olist_project.bronze.customers")
items = spark.table("olist_project.bronze.order_items")

###   **To preview Bronze tables **

In [0]:
%sql
SHOW TABLES IN olist_project.bronze;

### Display first 10 rows

In [0]:
display(orders.limit(10))
display(customers.limit(10))
display(items.limit(10))

### Display **schema**

In [0]:
orders.printSchema()
customers.printSchema()
items.printSchema()

### To check row counts

In [0]:
print("Orders rows:", orders.count())
print("Customers rows:", customers.count())
print("Order items rows:", items.count())

### Check missing values

In [0]:
from pyspark.sql.functions import col, sum

def count_nulls(df):
    return df.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])

display(count_nulls(orders))
display(count_nulls(customers))
display(count_nulls(items))

In [0]:
olist_joined = orders \
    .join(customers, "customer_id", "left") \
    .join(items, "order_id", "left")

In [0]:
display(olist_joined.limit(10))

In [0]:
from pyspark.sql.functions import col, sum, count

nulls_by_status = olist_joined.groupBy("order_status").agg(
    count("*").alias("total_rows"),
    sum(col("order_approved_at").isNull().cast("int")).alias("missing_approved_date"),
    sum(col("order_delivered_carrier_date").isNull().cast("int")).alias("missing_carrier_delivery_date"),
    sum(col("order_delivered_customer_date").isNull().cast("int")).alias("missing_customer_delivery_date"),
    sum(col("order_item_id").isNull().cast("int")).alias("missing_order_item"),
    sum(col("product_id").isNull().cast("int")).alias("missing_product"),
    sum(col("price").isNull().cast("int")).alias("missing_price"),
    sum(col("freight_value").isNull().cast("int")).alias("missing_freight")
).orderBy("order_status")

display(nulls_by_status)

In [0]:
olist_clean = olist_joined.filter(
    col("order_status") == "delivered"
)

In [0]:
from pyspark.sql.functions import col, sum

display(
    olist_clean.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in olist_clean.columns
    ])
)

### Drop rows with missing values needed for analysis

## Missing Value Handling

After checking missing values by `order_status`, I found that most missing delivery-related values were connected to orders that were not completed, such as canceled, unavailable, or shipped orders. Since this analysis focuses on completed sales and delivery performance, I filtered the dataset to keep only records where `order_status = 'delivered'`.

After this filter, only a very small number of important delivery fields were still missing. I dropped rows with missing `order_delivered_customer_date`, `order_estimated_delivery_date`, `price`, or `freight_value`, because these columns are required to calculate delivery time, delivery status, and total item value. This approach keeps the analysis consistent and avoids calculating delivery metrics for incomplete records.

In [0]:
olist_clean = olist_clean.dropna(
    subset=[
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "price",
        "freight_value"
    ]
)

### Add derived columns

In [0]:
from pyspark.sql.functions import col, to_timestamp, datediff, when, round

olist_silver = olist_clean \
    .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_delivered_customer_date", to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date")) \
    .withColumn("total_item_value", round(col("price") + col("freight_value"), 2)) \
    .withColumn(
        "delivery_days",
        datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
    ) \
    .withColumn(
        "delivery_status",
        when(col("order_delivered_customer_date") > col("order_estimated_delivery_date"), "Late")
        .otherwise("On time")
    )

### Preview Silver DataFrame

In [0]:
display(olist_silver.limit(10))

### Write to Silver Delta table

In [0]:
olist_silver.write.format("delta").mode("overwrite").saveAsTable(
    "olist_project.silver.orders_customers_items_clean"
)

### Confirm Silver table was created

In [0]:
display(spark.sql("SHOW TABLES IN olist_project.silver"))

### Read Silver table

In [0]:
silver_df = spark.table("olist_project.silver.orders_customers_items_clean")

display(silver_df.limit(10))

### Create temporary view for SQL analysis

In [0]:
silver_df.createOrReplaceTempView("olist_silver_view")

### Delivery and sales by state

In [0]:
gold_result = spark.sql("""
SELECT
    customer_state,
    delivery_status,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(total_item_value), 2) AS total_revenue,
    ROUND(AVG(delivery_days), 2) AS avg_delivery_days,
    ROUND(AVG(freight_value), 2) AS avg_freight_value
FROM olist_silver_view
GROUP BY customer_state, delivery_status
ORDER BY total_revenue DESC
""")

gold_result.write.format("delta").mode("overwrite").saveAsTable(
    "olist_project.gold.delivery_sales_by_state"
)

display(gold_result)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### Monthly sales trend

In [0]:
gold_monthly_sales_trend = spark.sql("""
SELECT
    DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM') AS order_month,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(total_item_value), 2) AS total_revenue,
    ROUND(AVG(total_item_value), 2) AS avg_item_value,
    ROUND(AVG(freight_value), 2) AS avg_freight_value
FROM olist_silver_view
GROUP BY DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM')
ORDER BY order_month
""")

gold_monthly_sales_trend.write.format("delta").mode("overwrite").saveAsTable(
    "olist_project.gold.monthly_sales_trend"
)

display(gold_monthly_sales_trend)

Databricks visualization. Run in Databricks to view.

### Late delivery rate by state

In [0]:
gold_late_delivery_by_state = spark.sql("""
SELECT
    customer_state,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(CASE WHEN delivery_status = 'Late' THEN 1 ELSE 0 END) AS late_orders,
    ROUND(SUM(CASE WHEN delivery_status = 'Late' THEN 1 ELSE 0 END) * 100.0 / COUNT(DISTINCT order_id), 2) AS late_delivery_percentage,
    ROUND(AVG(delivery_days), 2) AS avg_delivery_days
FROM olist_silver_view
GROUP BY customer_state
ORDER BY late_delivery_percentage DESC
""")

gold_late_delivery_by_state.write.format("delta").mode("overwrite").saveAsTable(
    "olist_project.gold.late_delivery_by_state"
)

display(gold_late_delivery_by_state)

Databricks visualization. Run in Databricks to view.

### Freight cost by state

In [0]:
gold_freight_by_state = spark.sql("""
SELECT
    customer_state,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(freight_value), 2) AS total_freight_cost,
    ROUND(AVG(freight_value), 2) AS avg_freight_value,
    ROUND(SUM(total_item_value), 2) AS total_revenue,
    ROUND(AVG(freight_value / total_item_value), 4) AS avg_freight_to_value_ratio
FROM olist_silver_view
GROUP BY customer_state
ORDER BY avg_freight_value DESC
""")

gold_freight_by_state.write.format("delta").mode("overwrite").saveAsTable(
    "olist_project.gold.freight_by_state"
)

display(gold_freight_by_state)

Databricks visualization. Run in Databricks to view.

### Seller performance summary

In [0]:
gold_seller_performance = spark.sql("""
SELECT
    seller_id,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT product_id) AS unique_products_sold,
    ROUND(SUM(total_item_value), 2) AS total_revenue,
    ROUND(AVG(total_item_value), 2) AS avg_item_value,
    ROUND(AVG(delivery_days), 2) AS avg_delivery_days
FROM olist_silver_view
GROUP BY seller_id
ORDER BY total_revenue DESC
""")

gold_seller_performance.write.format("delta").mode("overwrite").saveAsTable(
    "olist_project.gold.seller_performance_summary"
)

display(gold_seller_performance.limit(10))

Databricks visualization. Run in Databricks to view.

### Confirm all Gold tables

In [0]:
display(spark.sql("SHOW TABLES IN olist_project.gold"))

## Short Commentary

The data shows that most completed Olist orders were delivered on time. São Paulo (SP) generated the highest number of orders and the highest total revenue, which makes sense because it is one of the largest commercial regions in Brazil. I also noticed that states with higher sales volume usually have higher total freight costs because more products are being shipped there. Some states had longer average delivery times, which may suggest that location and logistics distance affect delivery performance. The missing values were mostly connected to orders that were not completed, such as canceled, unavailable, or shipped orders. This was useful because it helped justify filtering the dataset to only delivered orders. One thing that surprised me was that even after filtering to delivered orders, a very small number of delivery-related dates were still missing. Another interesting point was that late deliveries still existed even among completed orders, so delivery performance is not perfect. If I had more time, I would extend the project by adding the products, reviews, payments, sellers, and geolocation tables. This would allow me to analyze product-category performance and identify which categories generate the highest revenue, the largest number of orders, and the highest average freight costs. I would also join the reviews table to study whether late deliveries are associated with lower customer review scores or negative feedback. Adding the payments table would help analyze customer payment behavior, such as which payment methods are most common and whether installment payments are related to higher order values. Although I already used `seller_id` from the order items table, adding the sellers table would allow me to compare seller performance by seller location as well as revenue, number of orders, delivery speed, and late-delivery rate. Finally, I would include the geolocation table to explore how customer location and distance may affect freight cost, delivery time, and the probability of late delivery.